# Cálculo Diferencial e Integral — caderno de consulta e prática

Material de estudo orientado ao uso de Python para calcular, verificar e interpretar resultados de Cálculo Diferencial e Integral.

## Como usar
- Execute as células na ordem, especialmente a seção de configuração.
- Use `SymPy` para cálculos simbólicos e `NumPy`/`Matplotlib` para aproximações e gráficos.
- Um resultado calculado não dispensa interpretar domínio, hipóteses e significado.
- Compare resultados simbólicos com aproximações numéricas ou gráficos quando isso ajudar a validar a interpretação.

**Instalação, se necessário:** `pip install sympy numpy matplotlib scipy`

## Índice
1. Configuração e funções
2. **Tema 2 — Limites, continuidade e assíntotas: resolução de exercícios**
   - escolha do bloco por tipo de exercício; funções por partes; continuidade; assíntotas; conferência
3. **Tema 3 — Derivadas: conceitos, propriedades e cálculos**
4. **Tema 4 — Derivadas: aplicações**
5. **Tema 5 — Integrais: conceitos, propriedades e técnicas de integração**
6. **Tema 6 — Integrais: aplicações**
7. Checklist de escolha da ferramenta e exercícios rápidos

## 1. Configuração

Execute esta seção uma vez antes de usar os exercícios. Ela importa as bibliotecas, define os símbolos e reúne as funções auxiliares utilizadas nos cálculos de limites, continuidade e assíntotas.

In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

x, h, a, C = sp.symbols("x h a C", real=True)
e, pi, inf = sp.exp(1), sp.pi, sp.oo
sp.init_printing()

def iguais(a, b):
    """Verifica se duas expressões simbólicas são equivalentes."""
    return sp.simplify(a - b) == 0

def formatar_limite_bilateral(esquerda, direita):
    """Devolve o limite comum ou informa que os limites laterais diferem."""
    return esquerda if iguais(esquerda, direita) else "não existe: os limites laterais são diferentes"

def valor_funcao(funcao, ponto):
    """Calcula o valor da função no ponto, quando definido."""
    try:
        valor = funcao.subs(x, ponto)
        return "não definido" if valor.has(sp.nan, sp.zoo) else valor
    except (AttributeError, TypeError, ValueError):
        return "não foi possível avaliar"

def ramo_lateral(funcao, ponto, lado):
    """Seleciona o ramo de Piecewise ativo imediatamente de um lado do ponto."""
    if not isinstance(funcao, sp.Piecewise):
        return funcao
    ponto = sp.sympify(ponto)
    escala = max(1.0, abs(float(sp.N(ponto))))
    amostra = ponto + (-sp.Float(escala*1e-7) if lado == "-" else sp.Float(escala*1e-7))
    for expressao, condicao in funcao.args:
        if condicao is True or condicao == sp.true:
            return expressao
        try:
            if condicao.subs(x, amostra) == sp.true:
                return expressao
        except (AttributeError, TypeError):
            pass
    raise ValueError(f"Não foi encontrado ramo ativo à {'esquerda' if lado == '-' else 'direita'} de x={ponto}.")

def limite_lateral(funcao, ponto, lado):
    """Calcula limite lateral, selecionando o ramo de Piecewise automaticamente."""
    return sp.limit(ramo_lateral(funcao, ponto, lado), x, ponto, dir=lado)

def ramo_no_infinito(funcao, sinal):
    """Seleciona o ramo ativo para x muito positivo ('+') ou negativo ('-')."""
    if not isinstance(funcao, sp.Piecewise):
        return funcao
    amostra = sp.Integer(10**8 if sinal == "+" else -(10**8))
    for expressao, condicao in funcao.args:
        if condicao is True or condicao == sp.true:
            return expressao
        try:
            if condicao.subs(x, amostra) == sp.true:
                return expressao
        except (AttributeError, TypeError):
            pass
    raise ValueError("Não foi encontrado ramo ativo nessa direção do infinito.")

def candidatos_transicao(funcao):
    """Extrai candidatos a transição das condições de Piecewise."""
    candidatos = set()
    if not isinstance(funcao, sp.Piecewise):
        return []
    for _, condicao in funcao.args:
        for relacao in condicao.atoms(sp.core.relational.Relational):
            equacao = sp.simplify(relacao.lhs-relacao.rhs)
            if equacao.has(x):
                try:
                    for solucao in sp.solve(sp.Eq(equacao, 0), x):
                        if solucao.is_real is not False and solucao.is_finite is not False:
                            candidatos.add(solucao)
                except (NotImplementedError, ValueError, TypeError):
                    pass
    return sorted(candidatos, key=lambda v: float(sp.N(v)))

def candidatos_denominador(funcao):
    """Encontra zeros reais candidatos dos denominadores dos ramos."""
    candidatos = set()
    expressoes = [expr for expr, _ in funcao.args] if isinstance(funcao, sp.Piecewise) else [funcao]
    for expr in expressoes:
        denominador = sp.factor(sp.denom(sp.together(expr)))
        if denominador == 1:
            continue
        try:
            for solucao in sp.solve(sp.Eq(denominador, 0), x):
                if solucao.is_real is not False and solucao.is_finite is not False:
                    candidatos.add(solucao)
        except (NotImplementedError, ValueError, TypeError):
            pass
    return sorted(candidatos, key=lambda v: float(sp.N(v)))

def analisar_assintotas(funcao):
    """Analisa assíntotas de funções racionais usuais e funções por partes."""
    verticais = []
    candidatos = sorted(set(candidatos_transicao(funcao)+candidatos_denominador(funcao)),
                        key=lambda v: float(sp.N(v)))
    for ponto in candidatos:
        try:
            le, ld = limite_lateral(funcao, ponto, "-"), limite_lateral(funcao, ponto, "+")
            if any(v in (inf, -inf) for v in (le, ld)):
                verticais.append({"x": ponto, "esquerda": le, "direita": ld})
        except (ValueError, TypeError, NotImplementedError):
            pass
    infinitas = []
    for sinal, direcao in (("+", inf), ("-", -inf)):
        try:
            ramo = ramo_no_infinito(funcao, sinal)
            limite = sp.limit(ramo, x, direcao)
            if limite.is_finite is True:
                infinitas.append({"direcao": direcao, "tipo": "horizontal", "reta": sp.Eq(sp.Symbol("y"), limite)})
                continue
            m = sp.limit(ramo/x, x, direcao)
            if m.is_finite is True and sp.simplify(m) != 0:
                b = sp.limit(ramo-m*x, x, direcao)
                if b.is_finite is True:
                    infinitas.append({"direcao": direcao, "tipo": "oblíqua", "reta": sp.Eq(sp.Symbol("y"), sp.expand(m*x+b))})
        except (ValueError, TypeError, NotImplementedError):
            pass
    return {"verticais": verticais, "infinitas": infinitas}

def mostrar_assintotas(resultado):
    """Imprime assíntotas identificadas."""
    print("ASSÍNTOTAS VERTICAIS")
    if resultado["verticais"]:
        for item in resultado["verticais"]:
            print(f"  x = {item['x']} (limite esquerdo: {item['esquerda']}; direito: {item['direita']})")
    else:
        print("  Nenhuma assíntota vertical identificada.")
    print("\nASSÍNTOTAS QUANDO x → ±∞")
    if resultado["infinitas"]:
        for item in resultado["infinitas"]:
            direcao = "+∞" if item["direcao"] == inf else "−∞"
            print(f"  x → {direcao}: assíntota {item['tipo']}: {item['reta']}")
    else:
        print("  Nenhuma assíntota horizontal ou oblíqua identificada.")

### Definição de funções e domínio

Defina a função em f no bloco abaixo. Use x como variável independente. Exemplos: x**2 para x², sp.sqrt(x) para √x, sp.log(x) para ln(x) e sp.Piecewise((x**2, x<0), (x, True)) para uma função por partes. Use pi, inf e e para π, infinito e número de Euler, respectivamente.

Uma expressão simplificada pode ocultar restrições do domínio original. Cancelar um fator não torna a função original definida onde o denominador era zero.

In [ ]:
# EXEMPLO DE DEFINIÇÃO
f = (x**2 - 1)/(x - 1)

print("Função:", f)
print("Zeros do denominador:", sp.solve(sp.denom(f), x))

## 2. Limites, continuidade e assíntotas

### Localize o exercício pelo que ele pede

| O enunciado pede... | Bloco |
|---|---|
| Limite em um ponto, bilateral ou lateral | **2.1** |
| Limite de função definida por partes | **2.2** |
| Limite no infinito de função por partes | **2.3** |
| Continuidade e valor de parâmetro | **2.4** |
| Assíntotas de uma função comum | **2.5** |
| Assíntotas de função por partes | **2.6** |
| Simplificar indeterminação e calcular limite | **2.7** |
| Comparar valores numéricos perto do ponto | **2.8** |

Em cada bloco, altere apenas as linhas sob **ENTRADAS** e execute a célula. As funções auxiliares já estão definidas na configuração.

**Sintaxe:** use ** para potências, inf para +∞ e -inf para −∞. Em sp.Piecewise, cada par tem a forma (expressão, condição); as condições são verificadas na ordem escrita. Use (expressão, True) para o caso restante.

### 2.1 Limite em um ponto

Insira a expressão em f e o ponto de aproximação. Para um limite lateral, defina lado como '-' (esquerda) ou '+' (direita). Deixe lado como None para calcular ambos os limites laterais e o bilateral.

In [ ]:
# ENTRADAS
f = (x**2 - 4)/(x - 2)
ponto = 2
lado = None  # None = bilateral; use '-' ou '+' para limite lateral

# RESULTADOS
if lado in ("-", "+"):
    resultado = sp.limit(f, x, ponto, dir=lado)
    print(f"Limite pela {'esquerda' if lado == '-' else 'direita'}: {resultado}")
else:
    esquerda = sp.limit(f, x, ponto, dir="-")
    direita = sp.limit(f, x, ponto, dir="+")
    print("Limite pela esquerda:", esquerda)
    print("Limite pela direita:", direita)
    print("Limite bilateral:", formatar_limite_bilateral(esquerda, direita))

### 2.2 Limites laterais de função definida por partes

Defina f usando sp.Piecewise e informe o ponto. O bloco seleciona o ramo usado em cada lado e calcula os limites. O valor da função no ponto é mostrado separadamente, pois não determina os limites laterais.

In [ ]:
# ENTRADAS
f = sp.Piecewise(
    (x**2, x <= 2),
    (3*x - 2, x > 2)
)
ponto = 2

# RESULTADOS
esquerda = limite_lateral(f, ponto, "-")
direita = limite_lateral(f, ponto, "+")
print("Função:", f)
print("Limite pela esquerda:", esquerda)
print("Limite pela direita:", direita)
print("Limite bilateral:", formatar_limite_bilateral(esquerda, direita))
print("Valor da função no ponto:", valor_funcao(f, ponto))

### 2.3 Limites no infinito de funções por partes

Defina f com sp.Piecewise. O bloco identifica o ramo ativo para valores muito positivos e negativos e calcula os limites em +∞ e −∞.

In [ ]:
# ENTRADAS
f = sp.Piecewise(
    ((2*x**2 + 1)/(x**2 + 3), x > 2),
    ((x + 1)/(x - 1), True)
)

# RESULTADOS
for direcao, sinal in ((inf, "+"), (-inf, "-")):
    ramo = ramo_no_infinito(f, sinal)
    print(f"Ramo usado quando x tende a {direcao}:", ramo)
    print(f"Limite quando x tende a {direcao}:", sp.limit(ramo, x, direcao))

### 2.4 Continuidade e parâmetro desconhecido

Para continuidade em x = ponto, os limites laterais e o valor da função nesse ponto devem ser iguais. Insira as expressões válidas à esquerda e à direita e o valor no ponto. Declare parâmetros desconhecidos com sp.symbols.

In [ ]:
# ENTRADAS
k = sp.symbols("k", real=True)
expressao_esquerda = x**2
expressao_direita = 3*x - 2
valor_no_ponto = k
ponto = 2

# RESULTADOS
L_esq = sp.limit(expressao_esquerda, x, ponto, dir="-")
L_dir = sp.limit(expressao_direita, x, ponto, dir="+")
print("Limite pela esquerda:", L_esq)
print("Limite pela direita:", L_dir)
if sp.simplify(L_esq - L_dir) != 0:
    print("Continuidade impossível: os limites laterais são diferentes.")
else:
    solucoes = sp.solve(sp.Eq(valor_no_ponto, L_esq), dict=True)
    print("Valor exigido para continuidade:", L_esq)
    print("Parâmetro(s) que garantem continuidade:", solucoes if solucoes else "nenhum parâmetro desconhecido foi encontrado")

### 2.5 Assíntotas de uma função

Insira a expressão em f. O bloco procura automaticamente candidatos a assíntotas verticais nos zeros reais do denominador e calcula assíntotas horizontais ou oblíquas em cada direção infinita. Descontinuidades removíveis não são classificadas como assíntotas verticais.

In [ ]:
# ENTRADAS
f = (2*x**2 + 1)/(x - 1)

# RESULTADOS
mostrar_assintotas(analisar_assintotas(f))

### 2.6 Assíntotas de funções por partes

Defina f com sp.Piecewise. O bloco detecta pontos de transição nas condições e zeros dos denominadores dos ramos, examina os limites laterais e calcula as assíntotas no infinito. A detecção é voltada a condições usuais com desigualdades e expressões racionais; confira casos com condições simbólicas ou domínios mais complexos.

In [ ]:
# ENTRADAS
f = sp.Piecewise(
    ((x**2 + 1)/(x - 2), x > 2),
    ((3*x + 1)/(x + 1), True)
)

# RESULTADOS
mostrar_assintotas(analisar_assintotas(f))

### 2.7 Indeterminações, fatoração e racionalização

Use este bloco quando a substituição direta produzir uma forma como 0/0 ou quando houver radicais a simplificar. Insira f e ponto. Para estudar uma expressão oscilatória, substitua f pela expressão desejada.

In [ ]:
# ENTRADAS
f = (sp.sqrt(x + 1) - 1)/x
ponto = 0

# RESULTADOS
print("Expressão original:", f)
print("Forma simplificada:", sp.simplify(f))
print("Fatoração (quando aplicável):", sp.factor(f))
esquerda = sp.limit(f, x, ponto, dir="-")
direita = sp.limit(f, x, ponto, dir="+")
print("Limite pela esquerda:", esquerda)
print("Limite pela direita:", direita)
print("Limite bilateral:", formatar_limite_bilateral(esquerda, direita))
print("Limite de x²·sen(1/x) em 0:", sp.limit(x**2 * sp.sin(1/x), x, 0))

### 2.8 Conferência numérica perto de um ponto

Informe f, ponto e as distâncias para aproximar x por cada lado. A tabela ajuda a visualizar o comportamento, mas não prova a existência do limite. Distâncias menores aproximam mais o ponto, mas podem aumentar erros numéricos.

In [ ]:
# ENTRADAS
f = (x**2 - 4)/(x - 2)
ponto = 2
distancias = [1e-1, 1e-2, 1e-3, 1e-4]

# RESULTADOS
print(f"Valores de f(x) perto de x = {ponto}:")
print(f"{'Distância':>12} | {'Pela esquerda':>18} | {'Pela direita':>18}")
print("-" * 55)
for d in distancias:
    valor_esq = sp.N(f.subs(x, sp.Float(ponto-d)), 10)
    valor_dir = sp.N(f.subs(x, sp.Float(ponto+d)), 10)
    print(f"{d:12.1e} | {str(valor_esq):>18} | {str(valor_dir):>18}")

## 3. Derivadas: cálculo e consulta

Use os blocos abaixo conforme o que o enunciado pede. **Na maioria dos exercícios, o bloco 3.1 é suficiente:** informe a função e, se necessário, o ponto e a maior ordem de derivação. Os campos opcionais podem ficar como `None` quando o exercício não pedir aquele cálculo.

| O enunciado pede... | Bloco |
|---|---|
| Derivada, segunda derivada, terceira derivada ou outras ordens; derivada em um ponto; fatoração e simplificação | 3.1 |
| Verificar continuidade e derivabilidade em um ponto | 3.2 |
| Determinar parâmetros para que uma função por partes seja derivável | 3.2.1 |
| Derivação de uma equação com x e y; derivada de função inversa | 3.3 |

**Sintaxe:** use `x**2` para (x^2), `sp.sin(x)) para seno, `sp.cos(x)) para cosseno, `sp.exp(x)) para exponencial e `sp.log(x)` para logaritmo natural. Para frações, use parênteses: `(x**2+1)/(x-1)`.

### 3.1 Derivadas de uma função — todas as ordens em um único bloco

**Use quando o enunciado pedir:** derivada primeira, segunda derivada, terceira derivada, derivada de ordem (n), valor da derivada em um ponto, simplificação ou fatoração da derivada.

**ENTRADAS**
- `f`: função que aparece no enunciado.
- `ordem_maxima`: maior ordem solicitada. Ex.: 1 para (f'), 2 para (f''), 3 para (f''').
- `ponto`: valor de x para calcular as derivadas naquele ponto. Deixe `None` se não pedir avaliação pontual.

O bloco calcula as derivadas de todas as ordens até o limite informado. Também apresenta as formas simplificada e fatorada. Se informar um ponto, calcula cada derivada nele. Assim, não é necessário procurar outro bloco para cada ordem.

In [ ]:
# ENTRADAS
f = x**3 - 5*x**2 + 2*x + 7
ordem_maxima = 3
ponto = 2  # Use None se o exercício não pedir derivada em um ponto

# RESULTADOS
if not isinstance(ordem_maxima, int) or ordem_maxima < 1:
    raise ValueError("ordem_maxima deve ser um inteiro maior ou igual a 1")

derivadas = {}
expressao = f

for ordem in range(1, ordem_maxima + 1):
    expressao = sp.diff(expressao, x)
    derivadas[f"ordem_{ordem}"] = {
        "expressao": expressao,
        "simplificada": sp.simplify(expressao),
        "fatorada": sp.factor(expressao),
        "valor_no_ponto": sp.simplify(expressao.subs(x, ponto)) if ponto is not None else "Não solicitado"
    }

print("Função original:", f)
for nome, resultado in derivadas.items():
    print(f"\\nDerivada {nome.split('_')[1]}:")
    print("  Expressão:   ", resultado["expressao"])
    print("  Simplificada:", resultado["simplificada"])
    print("  Fatorada:    ", resultado["fatorada"])
    if ponto is not None:
        print(f"  Em x = {ponto}:", resultado["valor_no_ponto"])

### 3.2 Continuidade e derivabilidade em um ponto

**Use quando o enunciado pedir:** verificar se uma função é contínua ou derivável em determinado ponto; comparar derivadas laterais; identificar ponto anguloso; analisar uma função definida por partes.

**ENTRADAS**
- `g`: função. Para vários trechos, use `sp.Piecewise`.
- `ponto`: ponto analisado.

**Como interpretar:** continuidade exige que os limites laterais coincidam com o valor da função. Derivabilidade exige, além da continuidade, que as derivadas laterais coincidam. Nos extremos do domínio, considere apenas o lado pertencente ao domínio.

In [ ]:
# ENTRADAS
g = sp.Abs(x)
ponto = 0

# RESULTADOS
valor = g.subs(x, ponto)
limite_esquerda = sp.limit(g, x, ponto, dir="-")
limite_direita = sp.limit(g, x, ponto, dir="+")
derivada_esquerda = sp.limit(
    (g - valor) / (x - ponto), x, ponto, dir="-"
)
derivada_direita = sp.limit(
    (g - valor) / (x - ponto), x, ponto, dir="+"
)

continua = (limite_esquerda == valor) and (limite_direita == valor)
derivavel = continua and (derivada_esquerda == derivada_direita)

print("Valor da função:", valor)
print("Limite pela esquerda:", limite_esquerda)
print("Limite pela direita:", limite_direita)
print("É contínua no ponto?", continua)
print("Derivada pela esquerda:", derivada_esquerda)
print("Derivada pela direita:", derivada_direita)
print("É derivável no ponto?", derivavel)

### 3.2.1 Determinar parâmetros para garantir derivabilidade

**Use quando o enunciado fornecer uma função definida por partes com parâmetros desconhecidos** e pedir os valores desses parâmetros para que a função seja derivável no ponto de transição.

**ENTRADAS**
- `g`: função definida com `sp.Piecewise`. Informe os trechos e suas condições.
- `ponto`: ponto em que a expressão da função muda.
- `parametros`: lista das incógnitas que o sistema deve determinar.

**O que o código faz:** identifica os trechos à esquerda e à direita do ponto, monta as condições de continuidade e de igualdade das derivadas laterais e resolve o sistema para os parâmetros informados. Se houver mais de uma solução, todas serão apresentadas. O procedimento considera um ponto de transição interior ao domínio, com trechos diferenciáveis em cada lado.

In [ ]:
# ENTRADAS — função definida por partes
m, p = sp.symbols("m p", real=True)
g = sp.Piecewise(
    (p*x**2 + 2, x < 2),
    (m*x + 1, True)
)

# ENTRADAS — ponto de transição e parâmetros desconhecidos
ponto = 2
parametros = [m, p]  # informe aqui todos os parâmetros que deseja determinar

# RESULTADOS — identificação dos trechos à esquerda e à direita
trecho_esquerdo = ramo_lateral(g, ponto, "-")
trecho_direito = ramo_lateral(g, ponto, "+")

# RESULTADOS — condições de continuidade
valor_no_ponto = g.subs(x, ponto)
limite_esquerda = sp.limit(trecho_esquerdo, x, ponto, dir="-")
limite_direita = sp.limit(trecho_direito, x, ponto, dir="+")

condicoes_continuidade = [
    sp.Eq(limite_esquerda, valor_no_ponto),
    sp.Eq(limite_direita, valor_no_ponto)
]

# RESULTADOS — condição de igualdade das derivadas laterais
derivada_esquerda = sp.limit(sp.diff(trecho_esquerdo, x), x, ponto, dir="-")
derivada_direita = sp.limit(sp.diff(trecho_direito, x), x, ponto, dir="+")

condicao_derivabilidade = sp.Eq(derivada_esquerda, derivada_direita)

# RESULTADOS — resolução do sistema
condicoes = condicoes_continuidade + [condicao_derivabilidade]
solucoes = sp.solve(condicoes, parametros, dict=True)

print("Trecho à esquerda:", trecho_esquerdo)
print("Trecho à direita:", trecho_direito)
print("\nCONDIÇÕES")
print("Continuidade pela esquerda:", condicoes_continuidade[0])
print("Continuidade pela direita:", condicoes_continuidade[1])
print("Igualdade das derivadas laterais:", condicao_derivabilidade)

print("\nSOLUÇÕES DOS PARÂMETROS")
if solucoes:
    for indice, solucao in enumerate(solucoes, start=1):
        print(f"Solução {indice}:", solucao)
else:
    print("Nenhuma solução foi encontrada para os parâmetros informados.")

# RESULTADOS — conferência das condições em cada solução
for indice, solucao in enumerate(solucoes, start=1):
    print(f"\nCONFERÊNCIA DA SOLUÇÃO {indice}")
    for condicao in condicoes:
        print("  ", sp.simplify(condicao.lhs.subs(solucao) - condicao.rhs.subs(solucao)) == 0)


### 3.3 Derivação implícita e relação inversa

**Use quando o enunciado apresentar uma equação que relaciona x e y** e pedir derivadas de y em função de x, derivadas da relação inversa (x em função de y), derivadas de ordem superior ou valores em um ponto.

Informe a relação uma única vez em `F`, colocando todos os termos no lado esquerdo e zero no lado direito. Exemplo: `x**2 + x*y + y**2 = 1` torna-se `F = x**2 + x*y + y**2 - 1`.

O bloco apresenta as expressões simbólicas das derivadas de y em relação a x e da relação inversa, x em relação a y.

**Entradas adicionais (opcionais):**
- `ordem_maxima`: maior ordem da derivada a calcular. Use 1 para a primeira derivada, 2 para a segunda etc.
- `ponto_x`: coordenada x quando o enunciado pedir o valor da derivada em um ponto.
- `ponto_y`: coordenada y do ponto, caso seja fornecida. Se ficar como `None` e `ponto_x` tiver um valor, o bloco tentará determinar y substituindo x na relação original. Se houver mais de uma solução possível, informe também `ponto_y`.

As expressões simbólicas são sempre exibidas primeiro. A avaliação numérica é feita depois, quando `ponto_x` for informado.

In [25]:
# ENTRADAS — relação do enunciado
y = sp.symbols("y", real=True)
F = x**2 + x*y + y**2 - 1

# ENTRADAS — cálculos adicionais (mantenha os valores padrão quando não forem solicitados)
ordem_maxima = 2  # 1 calcula a primeira derivada; 2 calcula até a segunda etc.
ponto_x = None   # coordenada x para avaliação numérica
ponto_y = None   # coordenada y; se None, o bloco tenta obtê-la pela relação F = 0

# RESULTADOS — validação das entradas
if not isinstance(ordem_maxima, int) or isinstance(ordem_maxima, bool) or ordem_maxima < 1:
    raise ValueError("ordem_maxima deve ser um inteiro maior ou igual a 1")

# RESULTADOS — derivadas de y em relação a x
derivadas_diretas = {
    ordem: sp.factor(sp.idiff(F, y, x, n=ordem))
    for ordem in range(1, ordem_maxima + 1)
}

print("DERIVADAS DE y EM RELAÇÃO A x")
for ordem, expressao in derivadas_diretas.items():
    simbolo = "dy/dx" if ordem == 1 else f"d^{ordem}y/dx^{ordem}"
    print(f"Ordem {ordem} ({simbolo}):", expressao)

# RESULTADOS — derivadas da relação inversa, x em função de y
derivadas_inversas = {
    ordem: sp.factor(sp.idiff(F, x, y, n=ordem))
    for ordem in range(1, ordem_maxima + 1)
}

print("\nDERIVADAS DE x EM RELAÇÃO A y (RELAÇÃO INVERSA)")
for ordem, expressao in derivadas_inversas.items():
    simbolo = "dx/dy" if ordem == 1 else f"d^{ordem}x/dy^{ordem}"
    print(f"Ordem {ordem} ({simbolo}):", expressao)

# RESULTADOS — avaliação numérica opcional
if ponto_x is not None:
    if ponto_y is None:
        solucoes_y = sp.solve(F.subs(x, ponto_x), y)
        solucoes_reais = [
            solucao for solucao in solucoes_y
            if solucao.is_real is True
        ]

        if len(solucoes_reais) != 1:
            raise ValueError(
                "Não foi possível determinar um único valor real para y. "
                "Informe também ponto_y."
            )
        ponto_y_calculado = solucoes_reais[0]
    else:
        ponto_y_calculado = ponto_y

    ponto = {x: ponto_x, y: ponto_y_calculado}
    print(f"\nPONTO UTILIZADO: x = {ponto_x}, y = {ponto_y_calculado}")

    print("Derivadas de y em relação a x:")
    for ordem, expressao in derivadas_diretas.items():
        print(f"  Ordem {ordem}:", sp.simplify(expressao.subs(ponto)))

    print("Derivadas de x em relação a y:")
    for ordem, expressao in derivadas_inversas.items():
        print(f"  Ordem {ordem}:", sp.simplify(expressao.subs(ponto)))

DERIVADAS DE y EM RELAÇÃO A x
Ordem 1 (dy/dx): -(2*x + y)/(x + 2*y)
Ordem 2 (d^2y/dx^2): -6*(x**2 + x*y + y**2)/(x + 2*y)**3

DERIVADAS DE x EM RELAÇÃO A y (RELAÇÃO INVERSA)
Ordem 1 (dx/dy): -(x + 2*y)/(2*x + y)
Ordem 2 (d^2x/dy^2): -6*(x**2 + x*y + y**2)/(2*x + y)**3

AVALIAÇÃO NO PONTO (2, 2)
Derivadas de y em relação a x:
  Ordem 1: -1
  Ordem 2: -1/3
Derivadas de x em relação a y:
  Ordem 1: -1
  Ordem 2: -1/3


## 4. Derivadas: aplicações
### Localize o exercício pelo que ele pede

| Tipo de exercício | Bloco |
|---|---|
| Retas tangente e normal | 4.1 |
| Taxas de variação e taxas relacionadas | 4.2 |
| Crescimento, extremos e testes de derivadas | 4.3 |
| Concavidade e inflexão | 4.4 |
| Otimização em intervalo fechado | 4.5 |
| Visualização da análise de uma função | 4.6 |

Altere as expressões e os valores indicados em cada célula de código e execute para obter os resultados.


**Como identificar o procedimento:** inclinação ou tangente → `f'(a)`; normal → inclinação `-1/f'(a)` quando `f'(a)≠0`; velocidade → derivada da posição; aceleração → segunda derivada; extremos → pontos críticos e comparação; concavidade → sinal de `f''`; otimização → modelar objetivo, domínio e restrições.

A derivada fornece candidatos, **não** garante extremos. Pontos críticos interiores ocorrem onde `f'=0` ou onde `f'` não existe (com `f` definida).

### 4.1 Retas tangente e normal

No ponto `x=a`, a tangente é `y=f(a)+f'(a)(x-a)`. Se a tangente for horizontal, a normal é vertical (`x=a`); se a derivada não existir, é necessário analisar o caso geométrico, sem aplicar automaticamente a fórmula.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x**2+1
a = sp.Integer(2)
m = sp.diff(f,x).subs(x,a)
tangente = sp.expand(f.subs(x,a)+m*(x-a))
normal = sp.expand(f.subs(x,a)-sp.Rational(1,1)/m*(x-a)) if m != 0 else sp.Eq(x,a)
{'ponto':(a,f.subs(x,a)), 'inclinacao':m, 'tangente_y':tangente, 'normal_y':normal}

### 4.2 Taxas de variação e taxas relacionadas

Se `s(t)` é posição, `v(t)=s'(t)` e `a(t)=s''(t)`. Em taxas relacionadas, escreva uma equação entre grandezas, derive em relação ao **tempo** e só então substitua os valores conhecidos. As unidades devem ser consistentes.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
t = sp.symbols('t', real=True)
s = 2*t**3-9*t**2+12*t
v, aceleracao = sp.diff(s,t), sp.diff(s,t,2)
# Área de círculo A=pi*r(t)^2; quando r=3 e dr/dt=2:
r_t = sp.Function('r')(t)
A = sp.pi*r_t**2
dA_dt = sp.diff(A,t)
{'velocidade':v,'aceleracao':aceleracao,
 'dA_dt':dA_dt,
 'taxa_area_no_instante':dA_dt.subs({r_t:3,sp.diff(r_t,t):2})}

### 4.3 Crescimento, extremos e testes de derivadas

O sinal de `f'` indica crescimento (`+`) ou decrescimento (`-`) em intervalos do domínio. O **teste da primeira derivada** classifica um ponto crítico quando há troca de sinal: `+→-` máximo local, `-→+` mínimo local. Pelo **teste da segunda derivada**, se `f'(c)=0` e `f''(c)>0`, há mínimo local; se `f''(c)<0`, máximo local; se `f''(c)=0`, o teste é inconclusivo.

**Teorema de Fermat:** em um extremo local interior onde a derivada existe, `f'(c)=0`. **Teorema de Weierstrass:** função contínua em intervalo fechado e limitado atinge máximo e mínimo globais. Para extremos globais, compare valores em pontos críticos, pontos não diferenciáveis e extremos do intervalo.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x**3-6*x**2+9*x+1
d1, d2 = sp.diff(f,x), sp.diff(f,x,2)
criticos = sp.solve(sp.Eq(d1,0),x)
analise = [{'x':c,'f(x)':f.subs(x,c),'f_segunda':d2.subs(x,c),
            'tipo':('minimo local' if d2.subs(x,c)>0 else 'maximo local' if d2.subs(x,c)<0 else 'inconclusivo')}
           for c in criticos]
{'primeira_derivada':d1,'segunda_derivada':d2,'pontos_criticos':analise,
 'intervalos_crescentes':sp.solve_univariate_inequality(d1>0,x,relational=False),
 'intervalos_decrescentes':sp.solve_univariate_inequality(d1<0,x,relational=False)}

### 4.4 Concavidade e inflexão

`f''>0` indica concavidade para cima; `f''<0`, para baixo. Um ponto de inflexão exige **mudança efetiva de concavidade**; apenas resolver `f''=0` produz candidatos.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x**3-6*x**2+9*x+1
d2 = sp.diff(f,x,2)
candidatos_inflexao = sp.solve(sp.Eq(d2,0),x)
{'segunda_derivada':d2,'candidatos':candidatos_inflexao,
 'concavidade_para_cima':sp.solve_univariate_inequality(d2>0,x,relational=False),
 'concavidade_para_baixo':sp.solve_univariate_inequality(d2<0,x,relational=False)}

### 4.5 Otimização em intervalo fechado

Procedimento: (1) formule a função objetivo; (2) determine o domínio admissível; (3) calcule a derivada; (4) encontre candidatos interiores, incluindo pontos não diferenciáveis; (5) compare com os extremos do intervalo. Exemplo: maximizar a área de um retângulo de perímetro 20, com lados `x` e `10-x`, para `0≤x≤10`.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
area = x*(10-x)
candidatos = [sp.Integer(0),sp.Integer(10)] + [
 c for c in sp.solve(sp.Eq(sp.diff(area,x),0),x) if c.is_real and 0<=c<=10]
valores = {c:area.subs(x,c) for c in candidatos}
{'funcao_objetivo':area,'candidatos_e_areas':valores,
 'maximo':max(valores.items(),key=lambda item:item[1]),
 'minimo':min(valores.items(),key=lambda item:item[1])}

### 4.6 Visualização da análise de uma função

Altere `f` e o intervalo de visualização para estudar outros exemplos. O gráfico ajuda a interpretar, mas não substitui a verificação algébrica de domínio, sinais e extremos.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x**3-6*x**2+9*x+1
xs = np.linspace(-0.5,4.5,500)
fn = sp.lambdify(x,f,'numpy')
criticos = [float(c) for c in sp.solve(sp.Eq(sp.diff(f,x),0),x) if c.is_real]
inflexoes = [float(c) for c in sp.solve(sp.Eq(sp.diff(f,x,2),0),x) if c.is_real]
plt.figure(figsize=(9,5))
plt.plot(xs,fn(xs),label='f(x)')
for c in criticos:
    plt.scatter([c],[float(f.subs(x,c))],marker='o',s=65,label=f'crítico x={c:g}')
for c in inflexoes:
    plt.scatter([c],[float(f.subs(x,c))],marker='x',s=90,label=f'inflexão x={c:g}')
plt.axhline(0,color='gray',linewidth=.7)
plt.grid(True);plt.legend();plt.xlabel('x');plt.ylabel('f(x)');plt.show()

## 5. Integrais: conceitos, propriedades e técnicas
### Localize o exercício pelo que ele pede

| Tipo de exercício | Bloco |
|---|---|
| Integrais imediatas e propriedades | 5.1 |
| Soma de Riemann e integral definida | 5.2 |
| Teorema Fundamental do Cálculo (TFC) | 5.3 |
| Substituição de variável | 5.4 |
| Integração por partes | 5.5 |
| Frações parciais | 5.6 |
| Verificação e limitações | 5.7 |

Altere as expressões e os valores indicados em cada célula de código e execute para obter os resultados.


A **integral indefinida** fornece uma família de primitivas: se `F'=f`, então `∫f(x)dx=F(x)+C`. Uma **condição inicial** determina `C`. A **integral definida** produz um valor numérico (com sinal), interpretado como limite de somas de Riemann quando a função é integrável.

**Escolha do procedimento:** integral imediata → `sp.integrate`; função composta com derivada interna → substituição; produto de funções → partes; função racional → divisão polinomial e frações parciais. Verifique a derivada da primitiva e respeite o domínio.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
# Primitiva, constante de integração e condição inicial F(1)=5
f = 3*x**2+2*x
F0 = sp.integrate(f,x)
C = sp.symbols('C')
constante = sp.solve(sp.Eq((F0+C).subs(x,1),5),C)[0]
F = F0+constante
{'familia':F0+C,'constante':constante,'primitiva_particular':F,
 'verificacao':sp.simplify(sp.diff(F,x)-f)}

### 5.1 Integrais imediatas e propriedades

Linearidade: `∫(a f+b g)dx = a∫f dx+b∫g dx`. Primitivas úteis: `∫x**n dx=x**(n+1)/(n+1)+C` para `n≠-1`; `∫1/x dx=ln|x|+C` em intervalos sem zero; `∫exp(x)dx=exp(x)+C`; `∫sin(x)dx=-cos(x)+C`; `∫cos(x)dx=sin(x)+C`.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
integrandos = [x**4, 1/x, sp.exp(x), sp.sin(x), sp.cos(x), 2*x+sp.cos(x)]
[(g,sp.integrate(g,x),sp.simplify(sp.diff(sp.integrate(g,x),x)-g)) for g in integrandos]

### 5.2 Soma de Riemann e integral definida

Particione `[a,b]` em `n` intervalos de largura `Δx=(b-a)/n`. A soma à direita é `Σ f(a+iΔx)Δx`; quando converge com a malha tendendo a zero, obtém-se a integral. **A integral pode ser negativa** em regiões abaixo do eixo; área geométrica não é a mesma coisa.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x**2
a,b = sp.Integer(0),sp.Integer(2)
n,i = sp.symbols('n i',integer=True,positive=True)
dx = (b-a)/n
soma = sp.summation(f.subs(x,a+i*dx)*dx,(i,1,n))
{'soma_direita':sp.factor(soma),
 'limite':sp.limit(soma,n,sp.oo),
 'integral_exata':sp.integrate(f,(x,a,b))}

### 5.3 Teorema Fundamental do Cálculo (TFC)

Se `f` é contínua, a função `G(x)=∫_a^x f(t)dt` satisfaz `G'(x)=f(x)` (primeira parte). Se `F'=f`, então `∫_a^b f(x)dx=F(b)-F(a)` (segunda parte). Para limites variáveis, combine o TFC com a regra da cadeia.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
t = sp.symbols('t',real=True)
f_t = t**2+sp.sin(t)
G = sp.integrate(f_t,(t,0,x))
F = sp.integrate(f_t,t)
{'funcao_acumulada':G,'derivada_da_acumulada':sp.simplify(sp.diff(G,x)),
 'integral_0_a_2':sp.simplify(F.subs(t,2)-F.subs(t,0)),
 'conferencia':sp.integrate(f_t,(t,0,2)),
 'limite_variavel_x2':sp.simplify(sp.diff(sp.integrate(f_t,(t,0,x**2)),x))}

### 5.4 Substituição de variável

Procure uma expressão interna `u=g(x)` cuja derivada apareça no integrando. Transforme também `dx`; em integrais definidas, altere os limites para `u(a)` e `u(b)`, ou retorne à variável original antes de avaliar.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
u = sp.symbols('u',real=True)
integrando = 2*x*sp.cos(x**2)
via_u = sp.integrate(sp.cos(u),u).subs(u,x**2)
{'substituicao_u':'x**2','primitiva':via_u,
 'verificacao':sp.simplify(sp.diff(via_u,x)-integrando),
 'definida':sp.integrate(integrando,(x,0,sp.sqrt(sp.pi))),
 'limites_transformados':sp.integrate(sp.cos(u),(u,0,sp.pi))}

### 5.5 Integração por partes

Use `∫u dv = uv - ∫v du`, derivada da regra do produto. É especialmente útil para produtos envolvendo polinômios com exponenciais, logaritmos ou trigonométricas. A escolha de `u` deve simplificar a integral restante.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
u = x
dv = sp.exp(x)
du = sp.diff(u,x)
v = sp.integrate(dv,x)
resultado_partes = sp.simplify(u*v-sp.integrate(v*du,x))
{'integrando':u*dv,'resultado':resultado_partes,
 'verificacao':sp.simplify(sp.diff(resultado_partes,x)-u*dv)}

### 5.6 Frações parciais

Para `P(x)/Q(x)`, verifique se o grau do numerador é menor que o do denominador; caso contrário, divida os polinômios antes. Fatore `Q` e decomponha em frações simples. O SymPy oferece `sp.apart`; a decomposição e a integração dependem do domínio e dos fatores reais/complexos.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
racional = (2*x+3)/(x**2+3*x+2)
decomposta = sp.apart(racional,x)
primitiva = sp.integrate(decomposta,x)
impropria = (x**3+1)/(x**2-1)
{'fracao':racional,'decomposicao':decomposta,
 'primitiva':primitiva,
 'verificacao':sp.simplify(sp.diff(primitiva,x)-racional),
 'divisao_antes_de_decompor':sp.div(x**3+1,x**2-1,x)}

### 5.7 Verificação e limitações

`sp.integrate` pode devolver uma expressão especial ou uma integral não avaliada. Quando necessário, aproxime numericamente com `sp.N` ou técnicas numéricas e verifique o domínio. A constante `C` é omitida pela API simbólica; acrescente-a na interpretação de integrais indefinidas.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
integrando = x**4-3*x+sp.cos(x)
F = sp.integrate(integrando,x)
{'primitiva':F,'residuo_apos_derivar':sp.simplify(sp.diff(F,x)-integrando),
 'integral_definida_exata':sp.integrate(sp.sqrt(1+x**4),(x,0,1)),
 'valor_aproximado':sp.N(sp.integrate(sp.sqrt(1+x**4),(x,0,1)),12)}

## 6. Integrais: aplicações
### Localize o exercício pelo que ele pede

| Tipo de exercício | Bloco |
|---|---|
| Área líquida, área geométrica e área entre curvas | 6.1 |
| Comprimento de arco | 6.2 |
| Área de superfície de revolução | 6.3 |
| Volumes por seções, discos e arruelas | 6.4 |
| Cascas cilíndricas e rotação em torno de outro eixo | 6.5 |
| Aproximação numérica e checagens | 6.6 |

Altere as expressões e os valores indicados em cada célula de código e execute para obter os resultados.


**Mapa rápido:** área com sinal → `∫f`; área geométrica → `∫|f|`; área entre curvas → `∫|f-g|`; comprimento de arco → `∫sqrt(1+(f')²)`; área de superfície de revolução → `2π∫(raio)ds`; volume por seções → `∫A(x)dx`; sólidos de revolução → discos, arruelas ou cascas cilíndricas.

Antes de integrar, **desenhe/identifique a região, o intervalo, o eixo de rotação e as unidades**. Divida intervalos quando curvas se cruzam ou mudam de sinal.

### 6.1 Área líquida, área geométrica e área entre curvas

A integral definida soma áreas **com sinal**. Para área física entre gráfico e eixo, integre `|f|` (ou separe nas raízes). Entre curvas, integre `|f-g|`, particionando nas interseções; quando `f≥g` em todo intervalo, basta `∫(f-g)`.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x-1
a,b = 0,3
liquida = sp.integrate(f,(x,a,b))
geometrica = sp.integrate(sp.Abs(f),(x,a,b))
superior,inferior = x,x**2
entre = sp.integrate(superior-inferior,(x,0,1))
{'area_com_sinal':liquida,'area_geometrica':geometrica,
 'intersecoes':sp.solve(sp.Eq(superior,inferior),x),
 'area_entre_curvas_0_a_1':entre}

### 6.2 Comprimento de arco

Para curva `y=f(x)` continuamente derivável em `[a,b]`, o comprimento é `L=∫_a^b sqrt(1+[f'(x)]²)dx`. A função comprimento acumulado desde `a` é `s(x)=∫_a^x sqrt(1+[f'(t)]²)dt`. Em curvas paramétricas, use `∫sqrt((dx/dt)²+(dy/dt)²)dt`.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x**sp.Rational(3,2)
elemento_arco = sp.sqrt(1+sp.diff(f,x)**2)
comprimento = sp.integrate(elemento_arco,(x,0,4))
t = sp.symbols('t',real=True)
comprimento_parametrico = sp.integrate(sp.sqrt(sp.diff(t,t)**2+sp.diff(t**2,t)**2),(t,0,1))
{'ds_dx':elemento_arco,'comprimento_0_a_4':sp.simplify(comprimento),
 'comprimento_parametrico':comprimento_parametrico}

### 6.3 Área de superfície de revolução

Ao girar `y=f(x)≥0` em torno do eixo **x**, a área lateral é `S=2π∫_a^b f(x)sqrt(1+[f'(x)]²)dx`. Em torno do eixo **y**, para curva parametrizada por `x`, o raio é `|x|`: `S=2π∫|x|sqrt(1+[f'(x)]²)dx`. A fórmula calcula a superfície gerada pela curva, não tampas de um sólido.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
f = x  # segmento de reta y=x, de x=0 a x=2
a,b = 0,2
ds = sp.sqrt(1+sp.diff(f,x)**2)
S_eixo_x = sp.simplify(2*sp.pi*sp.integrate(f*ds,(x,a,b)))
S_eixo_y = sp.simplify(2*sp.pi*sp.integrate(x*ds,(x,a,b)))
{'superficie_ao_redor_de_x':S_eixo_x,'superficie_ao_redor_de_y':S_eixo_y}

### 6.4 Volumes por seções, discos e arruelas

Se a seção transversal perpendicular ao eixo `x` tem área `A(x)`, então `V=∫_a^b A(x)dx`. Para revolução em torno do eixo `x`: **discos** `V=π∫R(x)²dx`; **arruelas** `V=π∫[R(x)²-r(x)²]dx`. Exija `R≥r≥0` no intervalo; os raios são **distâncias ao eixo**, não valores arbitrários das funções.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
# Região entre y=x e o eixo x, 0<=x<=2, girada em torno do eixo x
V_disco = sp.pi*sp.integrate(x**2,(x,0,2))
# Região entre y=2 e y=x, 0<=x<=2, girada em torno do eixo x
V_arruela = sp.pi*sp.integrate(2**2-x**2,(x,0,2))
# Volume por seções quadradas de lado x
V_secoes = sp.integrate(x**2,(x,0,2))
{'discos':V_disco,'arruelas':V_arruela,'secoes_quadradas':V_secoes}

### 6.5 Cascas cilíndricas e rotação em torno de outro eixo

Para girar em torno do eixo **y** uma região sob `f(x)≥0`, com `0≤a≤b`, as cascas têm raio `x` e altura `f(x)`: `V=2π∫_a^b x f(x)dx`. Para um eixo vertical `x=c`, o raio é `|x-c|`; certifique-se de que a região não gere sobreposição na montagem das cascas. Escolha discos/arruelas ou cascas conforme a orientação das fatias.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
# Região sob y=x, entre x=0 e x=2, girada em torno do eixo y
V_cascas = 2*sp.pi*sp.integrate(x*x,(x,0,2))
# Conferência por discos perpendiculares ao eixo y: raio externo 2, interno y
y = sp.symbols('y',real=True)
V_arruelas_em_y = sp.pi*sp.integrate(4-y**2,(y,0,2))
{'cascas':V_cascas,'arruelas_no_eixo_y':V_arruelas_em_y,
 'coincidem':sp.simplify(V_cascas-V_arruelas_em_y)==0}

### 6.6 Aproximação numérica e checagens

Quando a primitiva não é elementar, a integral definida pode ser avaliada numericamente. Compare resultados simbólicos e numéricos, confira unidades (`comprimento`, `área`, `volume`) e confirme que áreas e volumes geométricos não sejam negativos. Uma regra do trapézio simples é útil para funções amostradas, mas a precisão depende da suavidade e da discretização.

In [ ]:
# ENTRADAS — altere os valores e as expressões do exemplo
# RESULTADOS — execute a célula para calcular
def regra_trapezios(func,a,b,n=1000):
    xs = np.linspace(a,b,n+1)
    ys = np.asarray(func(xs),dtype=float)
    return np.trapezoid(ys,xs) if hasattr(np,'trapezoid') else np.trapz(ys,xs)
f = x**2
exato = sp.integrate(f,(x,0,2))
aproximado = regra_trapezios(lambda z:z**2,0,2,1000)
{'exato':exato,'aproximado':aproximado,'erro_absoluto':abs(aproximado-float(exato))}

## 7. Escolha rápida da ferramenta

| Objetivo | Código típico |
|---|---|
| Limite simbólico | `sp.limit(expr, x, a)` |
| Limite lateral | `sp.limit(expr, x, a, dir='+' ou '-')` |
| Limite no infinito | `sp.limit(expr, x, sp.oo)` ou `-sp.oo` |
| Simplificar/fatorar | `sp.simplify(expr)`, `sp.factor(expr)` |
| Derivar | `sp.diff(expr, x)` |
| Derivada de ordem 2 | `sp.diff(expr, x, 2)` |
| Integrar sem limites | `sp.integrate(expr, x)` |
| Integral definida | `sp.integrate(expr, (x, a, b))` |
| Avaliar expressão | `expr.subs(x, valor)` |
| Gerar função numérica | `sp.lambdify(x, expr, 'numpy')` |

## Erros frequentes
- Tratar `0/0` como se fosse o valor do limite.
- Supor que o limite existe sem comparar os laterais.
- Confundir `f(a)` com `lim f(x)`.
- Concluir que toda descontinuidade gera assíntota vertical.
- Esquecer o domínio (por exemplo, `ln(x)` exige `x>0` nos reais).
- Em otimização, considerar apenas `f'=0` e ignorar extremos do intervalo ou pontos não diferenciáveis.
- Interpretar integral definida como área geométrica sem considerar o sinal.
- Esquecer `+C` em integrais indefinidas.

## Exercícios rápidos para testar o fluxo

Execute as células e confira se os resultados fazem sentido. Tente prever o resultado antes de executar.

1. Calcule `lim (x**2 - 4)/(x - 2)` quando `x→2`.
2. Verifique se `f(x)=(x**2-1)/(x-1)` tem limite em `x=1` e se está definida nesse ponto.
3. Encontre a derivada de `x**2*sin(x)`.
4. Calcule `∫_0^3 2x dx`.
5. Calcule a área entre `y=x` e `y=x**2` em `[0,1]`.

In [ ]:
# 1 e 2 — limites
L1 = sp.limit((x**2 - 4)/(x - 2), x, 2)
L2 = sp.limit((x**2 - 1)/(x - 1), x, 1)
valor_no_ponto = ((x**2 - 1)/(x - 1)).subs(x, 1)
L1, L2, valor_no_ponto

In [ ]:
# 3 a 5 — derivada e integrais
D3 = sp.diff(x**2 * sp.sin(x), x)
I4 = sp.integrate(2*x, (x, 0, 3))
I5 = sp.integrate(x - x**2, (x, 0, 1))
D3, I4, I5

## Síntese

Limites descrevem o comportamento de uma função perto de um ponto ou no infinito. Derivadas modelam taxas de variação instantânea e ajudam a analisar tangentes, crescimento, extremos e otimização. Integrais representam acumulação e permitem calcular áreas, deslocamentos, valores médios e volumes.

Ao resolver um problema computacionalmente, identifique a expressão e suas variáveis, verifique domínio e hipóteses, escolha a operação simbólica ou numérica adequada e interprete o resultado no contexto.